# Compile Weighted Pre-Fight Career Stats

This notebook reads raw UFC fight data, builds one row of fighter history for each upcoming fight, and writes both weighted and unweighted csv files. Performance statistics are written as exponentially weighted averages over prior fights (EWM(span=3)) in `w_pre_fight_stats.csv` and cumulative arithmetic averages in `pre_fight_stats.csv`; records, streaks, and other totals are calculated from prior fights only.

- Input: data_raw/`ufc_fights.csv`
- Outputs: data_clean/`w_pre_fight_stats.csv` and data_clean/`uw_pre_fight_stats.csv`

## 1. Load the raw fights

Read `data_raw/ufc_fights.csv`, assign a fight ID, parse dates, and define the output path.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data_clean").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_CLEAN_DIR = REPO_ROOT / "data_clean"
DATA_RAW_DIR = REPO_ROOT / "data_raw"

input_path = DATA_RAW_DIR / "ufc_fights.csv"

output_path = DATA_CLEAN_DIR / "w_pre_fight_stats.csv"
unweighted_output_path = DATA_CLEAN_DIR / "uw_pre_fight_stats.csv"

raw_fights = pd.read_csv(input_path)
raw_fights["fight_id"] = np.arange(len(raw_fights))
raw_fights["date"] = pd.to_datetime(raw_fights["date"], errors="coerce")

print("Loaded:", input_path)
print("Shape:", raw_fights.shape)
raw_fights.head()


Loaded: e:\Libraries\Documents\GitHub\UFC-Predictions\data_raw\ufc_fights.csv
Shape: (8763, 75)


,event_name,event_url,fight_url,date,location,attendance,fight_type,winner,R_fighter,B_fighter,...,B_BODY_att,B_LEG_landed,B_LEG_att,B_DISTANCE_landed,B_DISTANCE_att,B_CLINCH_landed,B_CLINCH_att,B_GROUND_landed,B_GROUND_att,fight_id
0,UFC Fight Night: Du Plessis vs. Usman,http://ufcstats.com/event-details/f354c50b8d63...,http://ufcstats.com/fight-details/52ddf20a1089...,2026-07-18,"Oklahoma City, Oklahoma, USA",NaN,Women's Strawweight Bout,Fatima Kline,Tabatha Ricci,Fatima Kline,...,28,5,6,56,151,9,12,1,1,0
1,UFC Fight Night: Du Plessis vs. Usman,http://ufcstats.com/event-details/f354c50b8d63...,http://ufcstats.com/fight-details/c64c6ba6eb1e...,2026-07-18,"Oklahoma City, Oklahoma, USA",NaN,Featherweight Bout,Tommy McMillen,Tommy McMillen,Alberto Montes,...,8,4,5,56,147,0,0,0,0,1
2,UFC Fight Night: Du Plessis vs. Usman,http://ufcstats.com/event-details/f354c50b8d63...,http://ufcstats.com/fight-details/3bd159c1bed1...,2026-07-18,"Oklahoma City, Oklahoma, USA",NaN,Featherweight Bout,Jose Delgado,Austin Bashi,Jose Delgado,...,11,4,4,29,78,5,6,1,1,2
3,UFC Fight Night: Du Plessis vs. Usman,http://ufcstats.com/event-details/f354c50b8d63...,http://ufcstats.com/fight-details/f2eb569176e4...,2026-07-18,"Oklahoma City, Oklahoma, USA",NaN,Welterweight Bout,Jean-Paul Lebosnoyani,Jean-Paul Lebosnoyani,Seokhyeon Ko,...,11,4,4,61,163,2,2,3,3,3
4,UFC Fight Night: Du Plessis vs. Usman,http://ufcstats.com/event-details/f354c50b8d63...,http://ufcstats.com/fight-details/46d800ce45fd...,2026-07-18,"Oklahoma City, Oklahoma, USA",NaN,Light Heavyweight Bout,Felipe Franco,Levi Rodrigues Jr.,Felipe Franco,...,1,0,0,12,20,0,0,27,36,4


## 2. Define conversion and labeling helpers

Create reusable functions for converting times, heights, reach, percentages, scheduled rounds, ages, winner sides, and finish-method labels into model-ready values.


In [2]:
# Helpers matching the old preprocessing conventions as closely as possible.

OLD_WIN_BY_COLUMNS = [
    "win_by_Decision_Majority",
    "win_by_Decision_Split",
    "win_by_Decision_Unanimous",
    "win_by_KO/TKO",
    "win_by_Submission",
    "win_by_TKO_Doctor_Stoppage",
]


def time_to_seconds(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    if value == "" or value.lower() == "nan":
        return np.nan
    parts = value.split(":")
    if len(parts) != 2:
        return np.nan
    try:
        return int(parts[0]) * 60 + int(parts[1])
    except ValueError:
        return np.nan


def height_to_cm(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    if value == "" or value.lower() == "nan" or value == "--":
        return np.nan
    try:
        if "'" in value:
            feet, inches = value.split("'", 1)
            inches = inches.replace('"', "").strip()
            return (int(feet) * 12 + int(inches)) * 2.54
        return float(value.replace('"', "")) * 2.54
    except ValueError:
        return np.nan


def reach_to_cm(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    if value == "" or value.lower() == "nan" or value == "--":
        return np.nan
    try:
        return float(value.replace('"', "")) * 2.54
    except ValueError:
        return np.nan


def numeric_percent(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        value = value.strip()
        if value.endswith("%"):
            return pd.to_numeric(value[:-1], errors="coerce") / 100
    return pd.to_numeric(value, errors="coerce")


def scheduled_rounds(value):
    if pd.isna(value):
        return np.nan
    value = str(value)
    if "Rnd" not in value:
        return np.nan
    try:
        return int(value.split("Rnd", 1)[0].strip())
    except ValueError:
        return np.nan


def fight_time_from_round(row):
    last_round = pd.to_numeric(row.get("last_round"), errors="coerce")
    last_round_time = time_to_seconds(row.get("last_round_time"))
    if pd.isna(last_round) or pd.isna(last_round_time):
        return np.nan
    return max((last_round - 1) * 5 * 60 + last_round_time, 0)


def calculate_age_at_fight(fight_date, dob):
    dob = pd.to_datetime(dob, errors="coerce")
    if pd.isna(fight_date) or pd.isna(dob):
        return np.nan
    return (fight_date - dob).days / 365.25


def winner_side(row):
    winner = "" if pd.isna(row["winner"]) else str(row["winner"]).strip()
    if winner == row["R_fighter"]:
        return "Red"
    if winner == row["B_fighter"]:
        return "Blue"
    return "Draw"


def old_win_by_name(value):
    value = "" if pd.isna(value) else str(value).strip()
    mapping = {
        "Decision - Majority": "win_by_Decision_Majority",
        "Decision - Split": "win_by_Decision_Split",
        "Decision - Unanimous": "win_by_Decision_Unanimous",
        "KO/TKO": "win_by_KO/TKO",
        "Submission": "win_by_Submission",
        "TKO - Doctor's Stoppage": "win_by_TKO_Doctor_Stoppage",
    }
    return mapping.get(value)


def prefixed_avg_name(col):
    if col == "total_time_fought(seconds)":
        return col
    if col.startswith("opp_"):
        return col.replace("opp_", "avg_opp_", 1)
    return f"avg_{col}"


## 3. Build fighter-level fight rows

Represent every bout from both the Red and Blue fighter perspectives, identify each fighter's result, convert fight statistics, and create finish-method indicator columns.


In [3]:
# Convert each bout into Red and Blue fighter-history rows.

raw_fights["winner"] = raw_fights["winner"].fillna("").astype(str).str.strip()
raw_fights["Winner"] = raw_fights.apply(winner_side, axis=1)
raw_fights["title_bout"] = False
if "fight_type" in raw_fights.columns:
    raw_fights["title_bout"] = raw_fights["fight_type"].fillna("").astype(str).str.contains("Title Bout", case=False)
raw_fights["no_of_rounds"] = raw_fights["format"].apply(scheduled_rounds)
raw_fights["total_time_fought(seconds)"] = raw_fights.apply(fight_time_from_round, axis=1)
raw_fights["last_round"] = pd.to_numeric(raw_fights["last_round"], errors="coerce")

STAT_COLS = [
    "KD",
    "SIG_STR_pct",
    "TD_pct",
    "SUB_ATT",
    "REV",
    "SIG_STR_att",
    "SIG_STR_landed",
    "TOTAL_STR_att",
    "TOTAL_STR_landed",
    "TD_att",
    "TD_landed",
    "HEAD_att",
    "HEAD_landed",
    "BODY_att",
    "BODY_landed",
    "LEG_att",
    "LEG_landed",
    "DISTANCE_att",
    "DISTANCE_landed",
    "CLINCH_att",
    "CLINCH_landed",
    "GROUND_att",
    "GROUND_landed",
    "CTRL",
]


def build_side_rows(side):
    opp_side = "B" if side == "R" else "R"
    rows = raw_fights[[
        "fight_id",
        "date",
        "winner",
        "Winner",
        "win_by",
        "last_round",
        "title_bout",
        "total_time_fought(seconds)",
        f"{side}_fighter",
        f"{opp_side}_fighter",
        f"{side}_DOB",
        f"{side}_Height",
        f"{side}_Reach",
    ] + [f"{side}_{col}" for col in STAT_COLS] + [f"{opp_side}_{col}" for col in STAT_COLS]].copy()

    rename_cols = {
        f"{side}_fighter": "fighter_name",
        f"{opp_side}_fighter": "opponent",
        f"{side}_DOB": "DOB",
        f"{side}_Height": "Height",
        f"{side}_Reach": "Reach",
    }

    for col in STAT_COLS:
        own_name = "CTRL_time(seconds)" if col == "CTRL" else col
        opp_name = "opp_CTRL_time(seconds)" if col == "CTRL" else f"opp_{col}"
        rename_cols[f"{side}_{col}"] = own_name
        rename_cols[f"{opp_side}_{col}"] = opp_name

    rows = rows.rename(columns=rename_cols)
    rows["side"] = side
    return rows


fighter_fights = pd.concat([build_side_rows("R"), build_side_rows("B")], ignore_index=True)

fighter_fights["result"] = np.select(
    [
        fighter_fights["winner"].eq(fighter_fights["fighter_name"]),
        fighter_fights["Winner"].eq("Draw"),
    ],
    ["Win", "Draw"],
    default="Loss",
)
fighter_fights["winner_for_history"] = np.select(
    [fighter_fights["result"].eq("Win"), fighter_fights["result"].eq("Loss")],
    ["hero", "opp"],
    default="draw",
)

fighter_fights["Height_cms"] = fighter_fights["Height"].apply(height_to_cm)
fighter_fights["Reach_cms"] = fighter_fights["Reach"].apply(reach_to_cm)
fighter_fights["age"] = fighter_fights.apply(lambda row: calculate_age_at_fight(row["date"], row["DOB"]), axis=1)
fighter_fights["CTRL_time(seconds)"] = fighter_fights["CTRL_time(seconds)"].apply(time_to_seconds)
fighter_fights["opp_CTRL_time(seconds)"] = fighter_fights["opp_CTRL_time(seconds)"].apply(time_to_seconds)

for col in ["SIG_STR_pct", "TD_pct", "opp_SIG_STR_pct", "opp_TD_pct"]:
    fighter_fights[col] = fighter_fights[col].apply(numeric_percent)

for col in OLD_WIN_BY_COLUMNS:
    fighter_fights[col] = 0

winner_method_col = fighter_fights["win_by"].apply(old_win_by_name)
for col in OLD_WIN_BY_COLUMNS:
    fighter_fights[col] = ((fighter_fights["result"] == "Win") & (winner_method_col == col)).astype(int)

print("Fighter-fight shape:", fighter_fights.shape)
fighter_fights.head()


Fighter-fight shape: (17526, 73)


,fight_id,date,winner,Winner,win_by,last_round,title_bout,total_time_fought(seconds),fighter_name,opponent,...,winner_for_history,Height_cms,Reach_cms,age,win_by_Decision_Majority,win_by_Decision_Split,win_by_Decision_Unanimous,win_by_KO/TKO,win_by_Submission,win_by_TKO_Doctor_Stoppage
0,0,2026-07-18,Fatima Kline,Blue,Decision - Unanimous,3,False,900,Tabatha Ricci,Fatima Kline,...,opp,154.94,154.94,31.403149,0,0,0,0,0,0
1,1,2026-07-18,Tommy McMillen,Red,KO/TKO,3,False,809,Tommy McMillen,Alberto Montes,...,hero,182.88,187.96,28.588638,0,0,0,1,0,0
2,2,2026-07-18,Jose Delgado,Blue,Decision - Unanimous,3,False,900,Austin Bashi,Jose Delgado,...,opp,167.64,177.80,24.799452,0,0,0,0,0,0
3,3,2026-07-18,Jean-Paul Lebosnoyani,Red,Decision - Unanimous,3,False,900,Jean-Paul Lebosnoyani,Seokhyeon Ko,...,hero,180.34,182.88,27.471595,0,0,1,0,0,0
4,4,2026-07-18,Felipe Franco,Blue,KO/TKO,2,False,400,Levi Rodrigues Jr.,Felipe Franco,...,opp,182.88,193.04,29.754962,0,0,0,0,0,0


## 4. Calculate pre-fight history features

Sort each fighter's history chronologically and use only earlier rows. The weighted output receives `EWM(span=3)` performance averages, while the unweighted output receives cumulative arithmetic averages. Rounds, title bouts, win methods, records, and streaks are accumulated from prior fights in both outputs.

That means striking, takedown, submission, control, and similar performance averages emphasize recent fights roughly: 
- 50% most recent fight
- 25% previous fight
- 12.5% fight before that


In [4]:
# Old-style pre-fight history features.
# avg_* and avg_opp_* are EWM(span=3) averages over prior fights, not per-minute rates.

EWM_SPAN = 3

own_numeric_cols = [
    "KD",
    "SIG_STR_pct",
    "TD_pct",
    "SUB_ATT",
    "REV",
    "SIG_STR_att",
    "SIG_STR_landed",
    "TOTAL_STR_att",
    "TOTAL_STR_landed",
    "TD_att",
    "TD_landed",
    "HEAD_att",
    "HEAD_landed",
    "BODY_att",
    "BODY_landed",
    "LEG_att",
    "LEG_landed",
    "DISTANCE_att",
    "DISTANCE_landed",
    "CLINCH_att",
    "CLINCH_landed",
    "GROUND_att",
    "GROUND_landed",
    "CTRL_time(seconds)",
]

opp_numeric_cols = [f"opp_{col}" for col in own_numeric_cols if col != "CTRL_time(seconds)"] + ["opp_CTRL_time(seconds)"]
ewm_numeric_cols = own_numeric_cols + opp_numeric_cols + ["total_time_fought(seconds)"]

for col in ewm_numeric_cols + ["last_round"] + OLD_WIN_BY_COLUMNS:
    fighter_fights[col] = pd.to_numeric(fighter_fights[col], errors="coerce")

fighter_fights = fighter_fights.sort_values(["fighter_name", "date", "fight_id"]).reset_index(drop=True)

ewm_prior = fighter_fights.groupby("fighter_name", group_keys=False)[ewm_numeric_cols].apply(
    lambda group: group.ewm(span=EWM_SPAN, adjust=False).mean().shift(1)
)

unweighted_prior = fighter_fights.groupby("fighter_name", group_keys=False)[ewm_numeric_cols].apply(
    lambda group: group.expanding().mean().shift(1)
)

prior_stats = fighter_fights[["fight_id", "fighter_name", "opponent", "date", "Height_cms", "Reach_cms", "age"]].copy()
prior_stats["prior_fights"] = fighter_fights.groupby("fighter_name").cumcount()

for col in ewm_numeric_cols:
    prior_stats[prefixed_avg_name(col)] = ewm_prior[col]

prior_stats["total_rounds_fought"] = (
    fighter_fights.groupby("fighter_name")["last_round"].cumsum()
    .groupby(fighter_fights["fighter_name"])
    .shift(1)
    .fillna(0)
)
prior_stats["total_title_bouts"] = (
    fighter_fights["title_bout"].astype(int).groupby(fighter_fights["fighter_name"]).cumsum()
    .groupby(fighter_fights["fighter_name"])
    .shift(1)
    .fillna(0)
)

prior_win_by = (
    fighter_fights.groupby("fighter_name")[OLD_WIN_BY_COLUMNS].cumsum()
    .groupby(fighter_fights["fighter_name"])
    .shift(1)
    .fillna(0)
)
for col in OLD_WIN_BY_COLUMNS:
    prior_stats[col] = prior_win_by[col]


def prior_result_stats(group):
    current_win_streak = 0
    current_lose_streak = 0
    longest_win_streak = 0
    wins = 0
    losses = 0
    draw = 0
    rows = []

    for _, row in group.iterrows():
        rows.append({
            "current_win_streak": current_win_streak,
            "current_lose_streak": current_lose_streak,
            "longest_win_streak": longest_win_streak,
            "wins": wins,
            "losses": losses,
            "draw": draw,
        })

        result = row["winner_for_history"]
        if result == "hero":
            wins += 1
            current_win_streak += 1
            current_lose_streak = 0
            longest_win_streak = max(longest_win_streak, current_win_streak)
        elif result == "opp":
            losses += 1
            current_win_streak = 0
            current_lose_streak += 1
        else:
            draw += 1
            current_win_streak = 0
            current_lose_streak = 0

    return pd.DataFrame(rows, index=group.index)


result_stats = fighter_fights.groupby("fighter_name", group_keys=False).apply(prior_result_stats)
prior_stats = pd.concat([prior_stats, result_stats], axis=1)

history_feature_cols = [
    "avg_KD",
    "avg_opp_KD",
    "avg_SIG_STR_pct",
    "avg_opp_SIG_STR_pct",
    "avg_TD_pct",
    "avg_opp_TD_pct",
    "avg_SUB_ATT",
    "avg_opp_SUB_ATT",
    "avg_REV",
    "avg_opp_REV",
    "avg_SIG_STR_att",
    "avg_SIG_STR_landed",
    "avg_opp_SIG_STR_att",
    "avg_opp_SIG_STR_landed",
    "avg_TOTAL_STR_att",
    "avg_TOTAL_STR_landed",
    "avg_opp_TOTAL_STR_att",
    "avg_opp_TOTAL_STR_landed",
    "avg_TD_att",
    "avg_TD_landed",
    "avg_opp_TD_att",
    "avg_opp_TD_landed",
    "avg_HEAD_att",
    "avg_HEAD_landed",
    "avg_opp_HEAD_att",
    "avg_opp_HEAD_landed",
    "avg_BODY_att",
    "avg_BODY_landed",
    "avg_opp_BODY_att",
    "avg_opp_BODY_landed",
    "avg_LEG_att",
    "avg_LEG_landed",
    "avg_opp_LEG_att",
    "avg_opp_LEG_landed",
    "avg_DISTANCE_att",
    "avg_DISTANCE_landed",
    "avg_opp_DISTANCE_att",
    "avg_opp_DISTANCE_landed",
    "avg_CLINCH_att",
    "avg_CLINCH_landed",
    "avg_opp_CLINCH_att",
    "avg_opp_CLINCH_landed",
    "avg_GROUND_att",
    "avg_GROUND_landed",
    "avg_opp_GROUND_att",
    "avg_opp_GROUND_landed",
    "avg_CTRL_time(seconds)",
    "avg_opp_CTRL_time(seconds)",
    "prior_fights",
    "total_time_fought(seconds)",
    "total_rounds_fought",
    "total_title_bouts",
    "current_win_streak",
    "current_lose_streak",
    "longest_win_streak",
    "wins",
    "losses",
    "draw",
] + OLD_WIN_BY_COLUMNS + [
    "Height_cms",
    "Reach_cms",
]

prior_stats = prior_stats[["fight_id", "fighter_name"] + history_feature_cols + ["age"]]

# Build the unweighted counterpart from the same raw fight-level inputs.
# Keep all non-average history fields identical to the weighted dataset.
unweighted_stats = prior_stats.copy()
for col in ewm_numeric_cols:
    unweighted_stats[prefixed_avg_name(col)] = unweighted_prior[col]
unweighted_stats = unweighted_stats[["fight_id", "fighter_name"] + history_feature_cols + ["age"]]

prior_stats.head()


,fight_id,fighter_name,avg_KD,avg_opp_KD,avg_SIG_STR_pct,avg_opp_SIG_STR_pct,avg_TD_pct,avg_opp_TD_pct,avg_SUB_ATT,avg_opp_SUB_ATT,...,draw,win_by_Decision_Majority,win_by_Decision_Split,win_by_Decision_Unanimous,win_by_KO/TKO,win_by_Submission,win_by_TKO_Doctor_Stoppage,Height_cms,Reach_cms,age
0,1250,AJ Cunningham,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0.0,0.0,0.0,0.0,0.0,0.0,177.80,180.34,29.483915
1,710,AJ Cunningham,0.0,1.0,0.480,0.650,NaN,0.660,0.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,177.80,180.34,30.518823
2,2318,AJ Dobson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0.0,0.0,0.0,0.0,0.0,0.0,185.42,193.04,30.069815
3,1941,AJ Dobson,0.0,0.0,0.570,0.620,NaN,0.370,0.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,185.42,193.04,30.759754
4,1509,AJ Dobson,0.0,0.0,0.465,0.545,1.0,0.185,0.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,185.42,193.04,31.564682


## 5. Merge, clean, and save the fight-level dataset

Merge the Red and Blue prior-history records onto each fight, fill missing reach and numeric values, select one shared final column order, and save both `w_pre_fight_stats.csv` and `pre_fight_stats.csv`.


In [5]:
# Merge back to one fight-level row, then apply old-style missing value treatment.

base_cols = [
    "fight_id",
    "date",
    "R_fighter",
    "B_fighter",
    "winner",
    "Winner",
    "win_by",
    "title_bout",
]
if "fight_type" in raw_fights.columns:
    base_cols.insert(7, "fight_type")

fight_rows = raw_fights[base_cols].copy()
fight_rows["winner_side"] = fight_rows["Winner"].map({"Red": "R", "Blue": "B"}).fillna("Draw")

r_prior = prior_stats.rename(columns={
    "fighter_name": "R_fighter",
    "age": "R_age",
    **{col: f"R_{col}" for col in history_feature_cols},
})
b_prior = prior_stats.rename(columns={
    "fighter_name": "B_fighter",
    "age": "B_age",
    **{col: f"B_{col}" for col in history_feature_cols},
})

compiled_stats = fight_rows.merge(r_prior, on=["fight_id", "R_fighter"], how="left")
compiled_stats = compiled_stats.merge(b_prior, on=["fight_id", "B_fighter"], how="left")
compiled_stats = compiled_stats.sort_values(["date", "fight_id"]).reset_index(drop=True)

# The old notebook filled missing reach with height, then filled remaining numeric missing values with medians.
for side in ["R", "B"]:
    compiled_stats[f"{side}_Reach_cms"] = compiled_stats[f"{side}_Reach_cms"].fillna(compiled_stats[f"{side}_Height_cms"])

numeric_cols = compiled_stats.select_dtypes(include=[np.number]).columns
compiled_stats[numeric_cols] = compiled_stats[numeric_cols].fillna(compiled_stats[numeric_cols].median(numeric_only=True))
compiled_stats[numeric_cols] = compiled_stats[numeric_cols].fillna(0)

compiled_stats["date"] = compiled_stats["date"].dt.strftime("%Y-%m-%d")
compiled_stats["winner"] = compiled_stats["winner"].replace("", np.nan).fillna("Draw")
compiled_stats["win_by"] = compiled_stats["win_by"].replace("", np.nan).fillna("Draw")
if "fight_type" in compiled_stats.columns:
    compiled_stats["fight_type"] = compiled_stats["fight_type"].replace("", np.nan).fillna("Unknown")

metadata_cols = [col for col in [
    "fight_id",
    "date",
    "R_fighter",
    "B_fighter",
    "winner",
    "Winner",
    "winner_side",
    "win_by",
    "fight_type",
    "title_bout",
] if col in compiled_stats.columns]

final_cols = (
    metadata_cols
    + [f"B_{col}" for col in history_feature_cols]
    + [f"R_{col}" for col in history_feature_cols]
    + ["B_age", "R_age"]
)

compiled_stats = compiled_stats[final_cols]
compiled_stats.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Shape:", compiled_stats.shape)
compiled_stats.head()

# Repeat the same fight-level compilation for the unweighted averages.
unweighted_r_prior = unweighted_stats.rename(columns={
    "fighter_name": "R_fighter",
    "age": "R_age",
    **{col: f"R_{col}" for col in history_feature_cols},
})
unweighted_b_prior = unweighted_stats.rename(columns={
    "fighter_name": "B_fighter",
    "age": "B_age",
    **{col: f"B_{col}" for col in history_feature_cols},
})

unweighted_compiled_stats = fight_rows.merge(unweighted_r_prior, on=["fight_id", "R_fighter"], how="left")
unweighted_compiled_stats = unweighted_compiled_stats.merge(unweighted_b_prior, on=["fight_id", "B_fighter"], how="left")
unweighted_compiled_stats = unweighted_compiled_stats.sort_values(["date", "fight_id"]).reset_index(drop=True)

for side in ["R", "B"]:
    unweighted_compiled_stats[f"{side}_Reach_cms"] = unweighted_compiled_stats[f"{side}_Reach_cms"].fillna(unweighted_compiled_stats[f"{side}_Height_cms"])

unweighted_numeric_cols = unweighted_compiled_stats.select_dtypes(include=[np.number]).columns
unweighted_compiled_stats[unweighted_numeric_cols] = unweighted_compiled_stats[unweighted_numeric_cols].fillna(unweighted_compiled_stats[unweighted_numeric_cols].median(numeric_only=True))
unweighted_compiled_stats[unweighted_numeric_cols] = unweighted_compiled_stats[unweighted_numeric_cols].fillna(0)

unweighted_compiled_stats["date"] = unweighted_compiled_stats["date"].dt.strftime("%Y-%m-%d")
unweighted_compiled_stats["winner"] = unweighted_compiled_stats["winner"].replace("", np.nan).fillna("Draw")
unweighted_compiled_stats["win_by"] = unweighted_compiled_stats["win_by"].replace("", np.nan).fillna("Draw")
if "fight_type" in unweighted_compiled_stats.columns:
    unweighted_compiled_stats["fight_type"] = unweighted_compiled_stats["fight_type"].replace("", np.nan).fillna("Unknown")

unweighted_compiled_stats = unweighted_compiled_stats[final_cols]
unweighted_compiled_stats.to_csv(unweighted_output_path, index=False)

print("Saved:", unweighted_output_path)
print("Shape:", unweighted_compiled_stats.shape)
print("Schemas identical:", list(compiled_stats.columns) == list(unweighted_compiled_stats.columns))
unweighted_compiled_stats.head()


Saved: e:\Libraries\Documents\GitHub\UFC-Predictions\data_clean\w_pre_fight_stats.csv
Shape: (8763, 144)
Saved: e:\Libraries\Documents\GitHub\UFC-Predictions\data_clean\pre_fight_stats.csv
Shape: (8763, 144)
Schemas identical: True


,fight_id,date,R_fighter,B_fighter,winner,Winner,winner_side,win_by,fight_type,title_bout,...,R_win_by_Decision_Majority,R_win_by_Decision_Split,R_win_by_Decision_Unanimous,R_win_by_KO/TKO,R_win_by_Submission,R_win_by_TKO_Doctor_Stoppage,R_Height_cms,R_Reach_cms,B_age,R_age
0,8748,1994-03-11,Royce Gracie,Patrick Smith,Royce Gracie,Red,R,KO/TKO,UFC 2 Tournament Title Bout,True,...,0.0,0.0,0.0,0.0,0.0,0.0,185.42,185.42,30.535250,27.244353
1,8749,1994-03-11,Royce Gracie,Remco Pardoel,Royce Gracie,Red,R,Submission,Open Weight Bout,False,...,0.0,0.0,0.0,1.0,0.0,0.0,185.42,185.42,24.799452,27.244353
2,8750,1994-03-11,Royce Gracie,Jason DeLucia,Royce Gracie,Red,R,Submission,Open Weight Bout,False,...,0.0,0.0,0.0,1.0,1.0,0.0,185.42,185.42,24.629706,27.244353
3,8751,1994-03-11,Patrick Smith,Johnny Rhodes,Patrick Smith,Red,R,Submission,Open Weight Bout,False,...,0.0,0.0,0.0,0.0,0.0,0.0,187.96,187.96,30.045175,30.535250
4,8752,1994-03-11,Remco Pardoel,Orlando Wiet,Remco Pardoel,Red,R,KO/TKO,Open Weight Bout,False,...,0.0,0.0,0.0,0.0,0.0,0.0,190.50,190.50,28.377823,24.799452
